# SmartBag — coleta e CSV
Colete duas rodadas completas das sete situações, sem reiniciar o ESP32. Prepare cada situação antes de iniciar; pare a coleta antes de reposicionar a bag.


## 1. Pacotes


In [ ]:
%pip -q install influxdb3-python pandas numpy


In [ ]:
import numpy as np
import pandas as pd
from getpass import getpass
from influxdb_client_3 import InfluxDBClient3
from google.colab import files

FEATURES = ["temperatura", "umidade", "delta_distancia", "luz", "mov_max", "incl_max"]


## 2. Consultar com SQL
Configure o device e o intervalo de uma execução contínua. Use horários com fuso; início incluso e fim exclusivo. Não misture reinicializações: rodada volta a 1.


In [ ]:
INFLUX_URL = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_BUCKET = "IoTSensores"
DEVICE = "SmartBagEquipe01"
INICIO = pd.to_datetime(input("Início (ISO, com fuso): "), utc=True)
FIM = pd.to_datetime(input("Fim (ISO, com fuso): "), utc=True)
INFLUX_TOKEN = getpass("Token InfluxDB: ")

client = InfluxDBClient3(host=INFLUX_URL, token=INFLUX_TOKEN, database=INFLUX_BUCKET)
query = f"""
SELECT *
FROM smartbag_raw_adc_2026
WHERE time >= '{INICIO.isoformat()}' AND time < '{FIM.isoformat()}'
ORDER BY time
"""
df = client.query(query=query, language="sql").to_pandas()
client.close()
df = df[df["device"] == DEVICE].rename(columns={"time": "timestamp"})
display(df.head())


## 3. Rotular e limpar
O rótulo vem da situação que você executou, não de limites dos sensores. Confira a tabela; se uma coleta ficou incorreta, descarte o par rodada/situação. Removemos apenas linhas incompletas ou sem rótulo conhecido.


In [ ]:
ROTULOS = {
    "parada_fechada": "ENTREGA_OK",
    "transporte_normal": "ENTREGA_OK",
    "buraco": "ENTREGA_OK",
    "aberta_parada": "ENTREGA_OK",
    "aberta_movimento": "REVISAR_ENTREGA",
    "tombamento": "REVISAR_ENTREGA",
    "problema_termico": "REVISAR_ENTREGA",
}
df["rodada"] = df["rodada"].astype(int)
df["target"] = df["situacao"].map(ROTULOS)
df = df.reindex(columns=["timestamp", "device", "rodada", "situacao"] + FEATURES + ["target"])
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + ["target"])
# Exemplo: df = df[~((df.rodada == 1) & (df.situacao == "buraco"))]
print(f"{len(df)} amostras")
display(df.groupby(["rodada", "situacao", "target"]).size().rename("amostras").to_frame())


## 4. Salvar
Use este mesmo CSV nos dois notebooks de treinamento. Rodada é o grupo do split; somente as seis features entram no modelo.


In [ ]:
df.to_csv("smartbag_dataset.csv", index=False)
files.download("smartbag_dataset.csv")
